In [2]:
import os
import time
import glob
import traceback
import pandas as pd
from nbformat import read
from nbconvert.preprocessors import ExecutePreprocessor

def executar_notebook(caminho_notebook, timeout=600, kernel_name="python3", salvar_executado=False):
    inicio = time.perf_counter()

    try:
        with open(caminho_notebook, "r", encoding="utf-8") as f:
            nb = read(f, as_version=4)

        ep = ExecutePreprocessor(timeout=timeout, kernel_name=kernel_name)
        ep.preprocess(nb, {"metadata": {"path": os.path.dirname(caminho_notebook) or "."}})

        fim = time.perf_counter()

        if salvar_executado:
            nome_base, ext = os.path.splitext(caminho_notebook)
            caminho_saida = f"{nome_base}_executado{ext}"
            with open(caminho_saida, "w", encoding="utf-8") as f:
                import nbformat
                nbformat.write(nb, f)
        else:
            caminho_saida = None

        return {
            "notebook": caminho_notebook,
            "status": "sucesso",
            "tempo_segundos": round(fim - inicio, 6),
            "erro": None,
            "arquivo_executado": caminho_saida
        }

    except Exception as e:
        fim = time.perf_counter()
        return {
            "notebook": caminho_notebook,
            "status": "erro",
            "tempo_segundos": round(fim - inicio, 6),
            "erro": "".join(traceback.format_exception_only(type(e), e)).strip(),
            "arquivo_executado": None
        }

def executar_varios_notebooks(pasta, timeout=600, kernel_name="python3", salvar_executado=False):
    caminhos = glob.glob(os.path.join(pasta, "**", "*.ipynb"), recursive=True)

    resultados = []

    for caminho in caminhos:
        print(f"Executando: {caminho}")
        resultado = executar_notebook(
            caminho_notebook=caminho,
            timeout=timeout,
            kernel_name=kernel_name,
            salvar_executado=salvar_executado
        )
        resultados.append(resultado)

    df = pd.DataFrame(resultados)
    return df

# ==========================
# EXEMPLO DE USO
# ==========================
pasta_dos_notebooks = "./notebooks"

df_resultados = executar_varios_notebooks(
    pasta=pasta_dos_notebooks,
    timeout=600,
    kernel_name="python3",
    salvar_executado=False
)

print(df_resultados)

# salvar resultados
df_resultados.to_csv("resultados_execucao_notebooks.csv", index=False)
df_resultados.to_excel("resultados_execucao_notebooks.xlsx", index=False)

Empty DataFrame
Columns: []
Index: []
